[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-0/lab-0.1-rooflines.ipynb)

# LAB·0.1 · Rooflines by hand

**Hardware:** any machine for the estimates; a Colab TPU runtime for the measured half (Runtime → Change runtime type → TPU).

The whole lab is one loop, run twice: predict an op's floor latency from chip constants alone, then measure the real thing and explain every miss. By the end you should trust your own arithmetic more than your intuition.

Chip constants come from the [scaling book](https://jax-ml.github.io/scaling-book/tpus/); the same table drives the site's EX·01 roofline instrument.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp

print(jax.__version__)
print(jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
if not ON_TPU:
    print("No TPU runtime: the estimate half works anywhere; the measured half needs Runtime -> TPU.")

## Part 1 · The constants and the ridge

A chip has two speeds: how fast it computes and how fast it feeds. The ratio (FLOPs per second over bytes per second) is the ridge: ops whose arithmetic intensity sits left of it are memory-bound, right of it compute-bound.

In [ ]:
# scaling-book chip constants (bf16), retrieved 2026-07-26
CHIPS = {
    "v5e": {"flops": 1.97e14, "hbm_bw": 8.2e11},
    "v6e": {"flops": 9.20e14, "hbm_bw": 1.6e12},
}
for name, c in CHIPS.items():
    print(f"{name}: ridge = {c['flops'] / c['hbm_bw']:.0f} FLOPs/byte")

## Part 2 · Predict five ops

For each op: count FLOPs, count minimal HBM bytes (bf16 = 2 bytes per element, read inputs once, write outputs once), divide for intensity, then the floor latency is `max(flops / peak_flops, bytes / hbm_bw)`. Fill in the table before running anything.

In [ ]:
def predict(name, flops, bytes_moved, chip):
    c = CHIPS[chip]
    intensity = flops / bytes_moved
    ridge = c["flops"] / c["hbm_bw"]
    bound = "compute" if intensity >= ridge else "memory"
    floor_s = max(flops / c["flops"], bytes_moved / c["hbm_bw"])
    return {"op": name, "intensity": intensity, "bound": bound, "floor_us": floor_s * 1e6}

N = 4096
ops = [
    predict("matmul NxNxN",      2 * N**3,     2 * 3 * N**2, "v5e"),
    predict("matmul skinny 8xNxN", 2 * 8 * N**2, 2 * (8*N + N*N + 8*N), "v5e"),
    predict("elementwise add",   N**2,         2 * 3 * N**2, "v5e"),
    predict("softmax rows",      5 * N**2,     2 * 2 * N**2, "v5e"),
    predict("reduce_sum",        N**2,         2 * (N**2 + N), "v5e"),
]
for o in ops:
    print(f"{o['op']:>18}: {o['intensity']:8.1f} F/B  {o['bound']:>7}-bound  floor {o['floor_us']:9.1f} us")

## Part 3 · Measure

The harness below is the benchmarking habit the whole track uses: warmup, `block_until_ready`, median of N, and the chip stated with every number. Reconcile each measurement against your prediction; a miss under 2x is a pass, and every miss needs a sentence explaining it (clock throttling, layout, not actually hitting peak BW, fusion you didn't predict).

In [ ]:
def measure(fn, *args, reps=20):
    fn(*args).block_until_ready()  # compile + warm
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(*args).block_until_ready()
        times.append(time.perf_counter() - t0)
    return float(np.median(times)) * 1e6  # us

In [ ]:
results = []
if ON_TPU:
    key = jax.random.key(0)
    a = jax.random.normal(key, (N, N), jnp.bfloat16)
    b = jax.random.normal(key, (N, N), jnp.bfloat16)
    s = jax.random.normal(key, (8, N), jnp.bfloat16)

    cases = {
        "matmul NxNxN": (jax.jit(lambda x, y: x @ y), a, b),
        "matmul skinny 8xNxN": (jax.jit(lambda x, y: x @ y), s, b),
        "elementwise add": (jax.jit(lambda x, y: x + y), a, b),
        "softmax rows": (jax.jit(lambda x: jax.nn.softmax(x, axis=-1)), a),
        "reduce_sum": (jax.jit(lambda x: jnp.sum(x, axis=-1)), a),
    }
    for name, (fn, *args) in cases.items():
        us = measure(fn, *args)
        pred = next(o for o in ops if o["op"] == name)
        results.append({"op": name, "measured_us": round(us, 1),
                        "predicted_us": round(pred["floor_us"], 1),
                        "ratio": round(us / pred["floor_us"], 2)})
        print(f"{name:>18}: measured {us:9.1f} us  predicted {pred['floor_us']:9.1f} us  ratio {us / pred['floor_us']:5.2f}x")
else:
    print("Skipped: no TPU in this runtime.")

## Results blob

The last cell prints the record for `bench/results.json` and the site's compare exhibit. Paste it into the gate discussion when you attempt Gate 00.

In [ ]:
import json
chip = jax.devices()[0].device_kind if ON_TPU else "none"
print(json.dumps({"lab": "0.1", "chip": chip, "n": N, "results": results}, indent=1))